# Runbook 5.5.a: Input validation and sanitization

**Skill:** Mitigate Risk · **Maps to:** 5.5.A · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** Build the server-side validation gate described in the lesson: given a batch of submitted form
field values, classify each as SAFE or BLOCKED using **allowlist rules per field type** plus
detection of known **injection meta-characters/keywords**, the same two checks a real
input-validation layer runs before any value reaches a database query, HTML page, or shell.

**Lesson objective.** Explain why every application must treat user input as untrusted, distinguish allowlist from denylist validation, describe where validation must occur to be trustworthy, and identify how input validation prevents injection classes like SQL injection, cross-site scripting, and command injection.

### What comes in
`submissions`: `list[dict]`, each `{"field": str, "type": str, "value": str}`. `type` is one of
`"zip"`, `"email"`, `"quantity"`, `"username"`.

### What must come out
`list[str]` of the `value`s that are BLOCKED, in the same order they appeared in the input.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```
[{"field":"zip","type":"zip","value":"94107"}]                    ->  []
[{"field":"zip","type":"zip","value":"9410"}]                     ->  ["9410"]
[{"field":"u","type":"username","value":"admin'--"}]               ->  ["admin'--"]
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(submissions: list) -> list
```

#### Input (`submissions`)
`list[dict]`, each `{"field": str, "type": str, "value": str}`. `type` is one of `"zip"`,
`"email"`, `"quantity"`, `"username"`.

#### Output (`result`)
`list[str]`, the `value` of every submission that is **BLOCKED**, in the same order as the
input. A submission is blocked if either condition holds:
1. Its `value` fails the allowlist pattern for its `type` (server-side shape check).
2. Its `value` matches a known injection meta-character/keyword signature (checked
   independently of `type`, a value can match the type's shape and still be blocked).

#### Allowlist rules
- `zip`: exactly 5 digits.
- `email`: `local@domain.tld` shape (letters/digits/`._+-` local part, dotted domain).
- `quantity`: integer string from `1` to `99` inclusive (no leading zero, no `0`, no `100+`).
- `username`: 3-20 characters, letters/digits/underscore only.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Injection signatures include (case-insensitive): `--`, `;`, `<script`, `UNION SELECT`,
  `OR 1=1`, `' OR '`, `DROP TABLE`, `../` path traversal, and URL-encoded `%3B`/`%2F`.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

Fill in `solve` below. Run the cell (Shift+Enter) after each change.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
"""Starter code for 5.5.a. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""
from typing import Dict, List


def solve(submissions: List[Dict[str, str]]) -> List[str]:
    """Classify each submission against per-field allowlist rules and known injection
    meta-character signatures; return the `value`s that are BLOCKED, in input order.

    Args:
        submissions: list of {"field": str, "type": str, "value": str}. `type` is one of
            "zip", "email", "quantity", "username".
    Returns:
        list[str] of the `value`s that fail validation (blocked), preserving input order.
    """
    # TODO: implement per the problem statement above
    raise NotImplementedError("implement solve() — see the problem statement above")


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_all_valid_none_blocked():
    subs = [
        {"field": "zip", "type": "zip", "value": "94107"},
        {"field": "email", "type": "email", "value": "user@example.com"},
        {"field": "qty", "type": "quantity", "value": "5"},
    ]
    assert impl.solve(subs) == []


def test_bad_zip_blocked():
    subs = [{"field": "zip", "type": "zip", "value": "9410"}]
    assert impl.solve(subs) == ["9410"]


def test_sql_injection_blocked_even_if_type_matches_shape():
    subs = [{"field": "username", "type": "username", "value": "admin'--"}]
    assert impl.solve(subs) == ["admin'--"]


def test_xss_payload_blocked():
    subs = [{"field": "username", "type": "username", "value": "<script>x</script>"}]
    assert impl.solve(subs) == ["<script>x</script>"]


def test_order_preserved_and_only_blocked_returned():
    subs = [
        {"field": "zip", "type": "zip", "value": "94107"},
        {"field": "zip", "type": "zip", "value": "abcde"},
        {"field": "qty", "type": "quantity", "value": "150"},
        {"field": "email", "type": "email", "value": "a@b.com"},
    ]
    assert impl.solve(subs) == ["abcde", "150"]


def test_quantity_zero_and_out_of_range_blocked():
    subs = [
        {"field": "qty", "type": "quantity", "value": "0"},
        {"field": "qty", "type": "quantity", "value": "100"},
        {"field": "qty", "type": "quantity", "value": "99"},
    ]
    assert impl.solve(subs) == ["0", "100"]


def test_union_select_blocked_regardless_of_case():
    subs = [{"field": "search", "type": "username", "value": "foo union select 1"}]
    assert impl.solve(subs) == ["foo union select 1"]


def test_path_traversal_blocked():
    subs = [{"field": "file", "type": "username", "value": "../etc/passwd"}]
    assert impl.solve(subs) == ["../etc/passwd"]


def test_empty_input():
    assert impl.solve([]) == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve([{"field":"zip","type":"zip","value":"94107"}]),
     []),
    (lambda: solve([{"field":"zip","type":"zip","value":"9410"}]),
     ["9410"]),
    (lambda: solve([{"field":"u","type":"username","value":"admin'--"}]),
     ["admin'--"]),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it